In [2]:
import pandas as pd 

All_Products = pd.read_csv('Datasets/ProductData.csv')
All_Products.head(2)

,Product ID,Title,Description,Sizes,Colors,Images,Category,Sub-Category,Gender
0,5204557e-f59f-4099-994b-7632d9e1200a,PATENT STAMPED CROC JEAN JACKET,JEAN JACKET\nFULLY LINED\nSIGNATURE DENIM TOP ...,"['52', '52', '54', '54']","['BLACK', 'BIRCH', 'BLACK', 'BIRCH']",['https://cdn.media.amplience.net/i/tom_ford/L...,outwear,jackets,men/ women/
1,49bc94b6-1a9b-44d9-8249-8358a68b34c0,SILK BLEND WIDE RIB POLO,SHORT SLEEVE POLO\nWIDER RIB\nSILK YARN PROVID...,"['52', '54', '48', '50', '56', '58']","['BLACK', 'BLACK', 'BLACK', 'BLACK', 'BLACK', ...",['https://cdn.media.amplience.net/i/tom_ford/K...,topwear,t-shirts,men/ women/


# Embeddings Generatoin & Qdrant Saving

#### This [sentence-transformers](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) model: It maps sentences & paragraphs to a 384 dimensional dense vector space

In [3]:
# load Embedding Model
from sentence_transformers import SentenceTransformer

model = SentenceTransformer (
    'sentence-transformers/all-MiniLM-L6-v2' )

/Users/user/opt/anaconda3/lib/python3.9/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [4]:
from datasets import Dataset , DatasetDict

Product_Dataset = DatasetDict()
Product_Dataset['ProductData'] = Dataset.from_pandas(All_Products)

Product_Dataset

DatasetDict({
    ProductData: Dataset({
        features: ['Product ID', 'Title', 'Description', 'Sizes', 'Colors', 'Images', 'Category', 'Sub-Category', 'Gender'],
        num_rows: 8009
    })
})

In [5]:
from tqdm import tqdm


def generate_embeddings(split , split_name :str ,  batch_size = 32 ) :
  '''
  • Arg 
  split : Dataset Split it can be train or test or validation
  split_name : Name of the Split for description.. 
  batch_size : size of chunk you wanted to push into model for generating embeddings in batch

  • return 
  return the list of embeddings 
  
  '''
  embeddings = []
  split_len = len(split)


  with tqdm(total = split_len , desc = f'Generating Embeddings for {split_name}...') as pbar :

    for i in range(0 , split_len , batch_size) :

      batch_sentences = split['Description'][i : i+batch_size]
      batch_encoding = model.encode(batch_sentences)
      embeddings.extend(batch_encoding)
      pbar.update(len(batch_sentences))

  return embeddings


# generate embeddings for all Products 
for split in Product_Dataset.keys() : 
  embeddings = generate_embeddings(Product_Dataset[split] , split_name = split , batch_size = 512 ) 
  Product_Dataset[split] = Product_Dataset[split].add_column('Embeddings' ,embeddings)

Generating Embeddings for ProductData...: 100%|█| 8009/8009 [04:46<00:00, 27.96i


In [6]:
from qdrant_client import QdrantClient , models

qdrant_client = QdrantClient(
    url="https://de049af4-984d-45d1-9f1b-adaec7a6bf8f.us-east4-0.gcp.cloud.qdrant.io:6333",
    api_key="vYVEVYkB8wviFoWnA3ZsXL-j8sOtrgTyEARdS_DvFrhs-8hqde6Vjg",
)

# qdrant_client = QdrantClient(url = "http://localhost:6333") 

In [7]:
collection_name = 'ProductCollection' 

qdrant_client.create_collection(
    collection_name = collection_name , 
    vectors_config = models.VectorParams (
        size = model.get_sentence_embedding_dimension() ,
        distance = models.Distance.COSINE ,
    ),
)

True

In [8]:
points = [
    models.PointStruct( 
    id = row['Product ID'] ,
    vector = row.pop('Embeddings'),
    payload = row )
    
    for row in tqdm(Product_Dataset['ProductData'], desc="Processing rows", unit="row")
]


Processing rows: 100%|████████████████████| 8009/8009 [00:10<00:00, 756.03row/s]


In [9]:
qdrant_client.upload_points(
    collection_name=collection_name,
    points=points,
)